# Detección de phishing: análisis con Spark (RDDs)

Mismo dataset que el análisis con Pandas (**212.216 URLs**, `label = 1` legítima, `label = 0` phishing),
procesado con **PySpark** usando RDDs y el modelo map / reduceByKey.

## Configuración

PySpark 4 necesita **Java 17 o superior**. Por eso se instala `openjdk-17` y se apunta `JAVA_HOME` a esa versión.

In [1]:
!apt-get update -qq > /dev/null
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
!pip install -q pyspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
# Descarga del dataset desde Google Drive (compartido como "Cualquier persona con el enlace")
import gdown

if not os.path.exists("train.csv"):
    gdown.download(id="1MTwmishpQVBW_iaxbzZiWAVf-sfAsOIJ", output="train.csv", quiet=True)

In [3]:
# Importo solo lo necesario. Con "from pyspark.sql.functions import *" se pisan
# las funciones max y sum de Python, y después fallan dentro de las lambdas.
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("phishing").getOrCreate()

df = spark.read.csv("train.csv", header=True, inferSchema=True)
rdd = df.rdd.cache()   # cache: el RDD se recorre muchas veces en el notebook

print(f"Cantidad de URLs: {rdd.count()}")

Cantidad de URLs: 212216


In [4]:
def promedio_por_clase(columna):
    """Promedio de una columna para phishing y para legítimas, en una sola pasada por el RDD."""
    promedios = (
        rdd.map(lambda fila: (fila.label, (1, fila[columna])))
           .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
           .mapValues(lambda t: t[1] / t[0])
           .collectAsMap()
    )
    return promedios[0], promedios[1]   # (phishing, legítimas)


def mostrar_promedios(columnas, formato=".4f"):
    for columna in columnas:
        phishing, legitimas = promedio_por_clase(columna)
        print(f"{columna:28s} phishing: {phishing:{formato}}   legítimas: {legitimas:{formato}}")

## 1. Nivel de ofuscación: promedio de subdominios y proporción de phishing

In [5]:
conteo_ofuscacion = (
    rdd.map(lambda fila: (fila.HasObfuscation, 1))
       .reduceByKey(lambda a, b: a + b)
       .collect()
)
print(conteo_ofuscacion)

[(0, 211777), (1, 439)]


Hay muy pocas URLs con ofuscación (439), así que cualquier conclusión sobre este grupo tiene poco respaldo.
Para clasificarlas defino cuatro niveles según `ObfuscationRatio`.

In [6]:
def clasificar_ofuscacion(ratio):
    if ratio == 0:
        return "Sin ofuscación"
    elif ratio < 0.05:
        return "Bajo"
    elif ratio < 0.1:
        return "Medio"
    else:
        return "Alto"


# (nivel, (subdominios, es_phishing, 1))
resultados_ofuscacion = (
    rdd.map(lambda fila: (
            clasificar_ofuscacion(fila.ObfuscationRatio),
            (fila.NoOfSubDomain, 1 if fila.label == 0 else 0, 1),
        ))
       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2]))
       .mapValues(lambda t: (t[0] / t[2], t[1] / t[2], t[2]))   # (prom. subdominios, % phishing, total)
       .collect()
)

for nivel, (prom_sub, prop_phishing, total) in resultados_ofuscacion:
    print(f"{nivel:15s} subdominios: {prom_sub:.2f}   phishing: {prop_phishing:7.2%}   URLs: {total}")

Bajo            subdominios: 1.08   phishing: 100.00%   URLs: 165
Sin ofuscación  subdominios: 1.16   phishing:  42.67%   URLs: 211777
Medio           subdominios: 1.88   phishing: 100.00%   URLs: 196
Alto            subdominios: 0.77   phishing: 100.00%   URLs: 78


### Conclusión

| Nivel de ofuscación | Prom. subdominios | % Phishing | Cantidad de URLs |
|---|---|---|---|
| Sin ofuscación (ratio = 0) | 1,16 | 42,67% | 211.777 |
| Bajo (0 < ratio < 0,05) | 1,08 | 100% | 165 |
| Medio (0,05 ≤ ratio < 0,1) | 1,88 | 100% | 196 |
| Alto (ratio ≥ 0,1) | 0,77 | 100% | 78 |

Todas las URLs con algún nivel de ofuscación son de phishing. Aun así, son solo 439 contra más de 211.000
sin ofuscación, así que la asociación es fuerte en este dataset pero no alcanza para afirmar una regla general.

## 2. Tipo de caracteres predominante en la URL: largo promedio del dominio y proporción de phishing

Para cada URL comparo la cantidad de letras, dígitos y caracteres especiales
(`=`, `?`, `&` y otros). El tipo con mayor cantidad es el predominante; si hay empate, la URL es "mixta".

In [7]:
def tipo_predominante(fila):
    letras = fila.NoOfLettersInURL
    digitos = fila.NoOfDegitsInURL
    especiales = (fila.NoOfOtherSpecialCharsInURL + fila.NoOfEqualsInURL
                  + fila.NoOfQMarkInURL + fila.NoOfAmpersandInURL)

    maximo = max(letras, digitos, especiales)
    empatados = sum([letras == maximo, digitos == maximo, especiales == maximo])

    if empatados > 1:
        return "Mixta"
    elif letras == maximo:
        return "Letras"
    elif digitos == maximo:
        return "Dígitos"
    else:
        return "Especiales"


resultados_tipo = (
    rdd.map(lambda fila: (tipo_predominante(fila), (1, 1 if fila.label == 0 else 0, fila.DomainLength)))
       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2]))
       .collect()
)

for tipo, (cantidad, phishing, suma_largo) in resultados_tipo:
    print(f"{tipo:11s} URLs: {cantidad:7d}   phishing: {phishing / cantidad:7.2%}   largo prom. dominio: {suma_largo / cantidad:.2f}")

Letras      URLs:  209147   phishing:  42.00%   largo prom. dominio: 21.41
Mixta       URLs:     589   phishing:  91.00%   largo prom. dominio: 21.09
Dígitos     URLs:    2479   phishing:  97.54%   largo prom. dominio: 26.14
Especiales  URLs:       1   phishing: 100.00%   largo prom. dominio: 13.00


### Conclusión

| Predominan | Cantidad de URLs | % Phishing | Largo prom. del dominio |
|---|---|---|---|
| Letras | 209.147 | 42,00% | 21,41 |
| Mixta (empate) | 589 | 91,00% | 21,09 |
| Dígitos | 2.479 | 97,54% | 26,14 |
| Especiales | 1 | 100% | 13,00 |

- Las URLs donde predominan las letras son la gran mayoría y tienen la menor proporción de phishing.
- Cuando predominan los dígitos, el 97,5% son phishing: es una señal muy fuerte.
- Los empates también se inclinan claramente hacia phishing.
- Con una sola URL donde predominan los caracteres especiales, no se puede sacar conclusión de ese grupo.

## 3. URLs con pocos recursos (imágenes, CSS, JS): según campo de contraseña y botón de envío

In [8]:
# Distribución de la cantidad de recursos, para elegir un umbral
frecuencias = (
    rdd.map(lambda fila: (fila.NoOfImage + fila.NoOfCSS + fila.NoOfJS, 1))
       .reduceByKey(lambda a, b: a + b)
       .sortByKey()
       .take(15)
)
for recursos, cantidad in frecuencias:
    print(f"{recursos} recursos: {cantidad} URLs")

0 recursos: 62287 URLs
1 recursos: 6551 URLs
2 recursos: 7012 URLs
3 recursos: 2660 URLs
4 recursos: 2857 URLs
5 recursos: 1756 URLs
6 recursos: 1373 URLs
7 recursos: 1294 URLs
8 recursos: 1297 URLs
9 recursos: 1142 URLs
10 recursos: 1740 URLs
11 recursos: 1318 URLs
12 recursos: 1265 URLs
13 recursos: 1543 URLs
14 recursos: 1195 URLs


In [9]:
UMBRAL_RECURSOS = 3

resultados_recursos = (
    rdd.filter(lambda fila: fila.NoOfImage + fila.NoOfCSS + fila.NoOfJS < UMBRAL_RECURSOS)
       .map(lambda fila: (
            (fila.HasPasswordField, fila.HasSubmitButton),
            (1, 1 if fila.label == 0 else 0, fila.LineOfCode),
        ))
       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2]))
       .collect()
)

for (password, submit), (cantidad, phishing, lineas) in resultados_recursos:
    print(f"Password: {password}  Submit: {submit}  ->  URLs: {cantidad:6d}   "
          f"phishing: {phishing / cantidad:7.2%}   líneas prom.: {lineas / cantidad:.1f}")

Password: 0  Submit: 0  ->  URLs:  73405   phishing:  99.79%   líneas prom.: 35.4
Password: 1  Submit: 1  ->  URLs:    526   phishing:  98.86%   líneas prom.: 204.3
Password: 0  Submit: 1  ->  URLs:   1781   phishing:  93.71%   líneas prom.: 129.2
Password: 1  Submit: 0  ->  URLs:    138   phishing: 100.00%   líneas prom.: 85.3


### Conclusión (URLs con menos de 3 recursos)

| Campo de contraseña | Botón de envío | Cantidad de URLs | % Phishing | Prom. líneas de código |
|---|---|---|---|---|
| No | No | 73.405 | 99,79% | 35,4 |
| Sí | Sí | 526 | 98,86% | 204,3 |
| No | Sí | 1.781 | 93,71% | 129,2 |
| Sí | No | 138 | 100% | 85,3 |

- Las páginas livianas son casi todas phishing (más del 93% en todos los grupos):
  los atacantes tienden a usar páginas simples, sin imágenes, CSS ni JS.
- Las que tienen formularios tienen más líneas de código, pero siguen siendo mayoritariamente phishing.

## 4. Combinaciones de TLD y HTTPS (con al menos 50 URLs)

In [10]:
resultados_tld = (
    rdd.map(lambda fila: ((fila.TLD, fila.IsHTTPS), (1, fila.DomainLength, 1 if fila.label == 0 else 0)))
       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2]))
       .filter(lambda par: par[1][0] >= 50)
       .map(lambda par: (par[0], par[1][0], par[1][1] / par[1][0], par[1][2] / par[1][0]))
       .sortBy(lambda fila: (-fila[3], -fila[1]))   # más phishing primero; a igualdad, más URLs primero
)

def mostrar_tld(filas):
    for (tld, https), cantidad, largo, phishing in filas:
        print(f"TLD: {str(tld):10s} HTTPS: {'Sí' if https == 1 else 'No'}  ->  URLs: {cantidad:6d}   "
              f"largo prom. dominio: {largo:6.2f}   phishing: {phishing:7.2%}")

mostrar_tld(resultados_tld.collect())

TLD: com        HTTPS: No  ->  URLs:  20514   largo prom. dominio:  21.87   phishing: 100.00%
TLD: ru         HTTPS: No  ->  URLs:   2397   largo prom. dominio:  19.72   phishing: 100.00%
TLD: net        HTTPS: No  ->  URLs:   1844   largo prom. dominio:  21.04   phishing: 100.00%
TLD: org        HTTPS: No  ->  URLs:   1757   largo prom. dominio:  20.53   phishing: 100.00%
TLD: top        HTTPS: No  ->  URLs:   1492   largo prom. dominio:  34.42   phishing: 100.00%
TLD: cf         HTTPS: No  ->  URLs:   1012   largo prom. dominio:  16.65   phishing: 100.00%
TLD: ga         HTTPS: No  ->  URLs:    958   largo prom. dominio:  16.39   phishing: 100.00%
TLD: xyz        HTTPS: No  ->  URLs:    941   largo prom. dominio:  21.82   phishing: 100.00%
TLD: ml         HTTPS: No  ->  URLs:    855   largo prom. dominio:  18.11   phishing: 100.00%
TLD: co         HTTPS: No  ->  URLs:    717   largo prom. dominio:  23.80   phishing: 100.00%
TLD: info       HTTPS: No  ->  URLs:    640   largo prom. do

In [11]:
# Solo las combinaciones más grandes (al menos 1000 URLs), para leerlas más fácil
mostrar_tld(resultados_tld.filter(lambda fila: fila[1] >= 1000).collect())

TLD: com        HTTPS: No  ->  URLs:  20514   largo prom. dominio:  21.87   phishing: 100.00%
TLD: ru         HTTPS: No  ->  URLs:   2397   largo prom. dominio:  19.72   phishing: 100.00%
TLD: net        HTTPS: No  ->  URLs:   1844   largo prom. dominio:  21.04   phishing: 100.00%
TLD: org        HTTPS: No  ->  URLs:   1757   largo prom. dominio:  20.53   phishing: 100.00%
TLD: top        HTTPS: No  ->  URLs:   1492   largo prom. dominio:  34.42   phishing: 100.00%
TLD: cf         HTTPS: No  ->  URLs:   1012   largo prom. dominio:  16.65   phishing: 100.00%
TLD: dev        HTTPS: Sí  ->  URLs:   1881   largo prom. dominio:  33.43   phishing:  98.30%
TLD: app        HTTPS: Sí  ->  URLs:   5531   largo prom. dominio:  24.47   phishing:  97.76%
TLD: co         HTTPS: Sí  ->  URLs:   4144   largo prom. dominio:  32.92   phishing:  89.99%
TLD: io         HTTPS: Sí  ->  URLs:   3356   largo prom. dominio:  20.94   phishing:  88.11%
TLD: pl         HTTPS: Sí  ->  URLs:   1022   largo prom. do

In [12]:
# Proporción de phishing con y sin HTTPS
for https, (cantidad, phishing) in (
    rdd.map(lambda fila: (fila.IsHTTPS, (1, 1 if fila.label == 0 else 0)))
       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
       .collect()
):
    print(f"HTTPS {'Sí' if https == 1 else 'No'}: {cantidad} URLs, {phishing / cantidad:.2%} de phishing")

HTTPS No: 46167 URLs, 100.00% de phishing
HTTPS Sí: 166049 URLs, 26.88% de phishing


### Conclusión

- La falta de HTTPS es un indicador muy fuerte: todas las combinaciones sin HTTPS con suficientes URLs
  (.com, .org, .xyz, .online, etc.) tienen 100% de phishing. Solo en `.com` sin HTTPS hay más de 20.000 URLs.
- HTTPS sirve como **filtro negativo** (sin HTTPS, casi seguro es phishing),
  pero no como filtro positivo: tener HTTPS no garantiza que la página sea legítima.
- Los TLDs con más phishing tienden a tener dominios algo más largos.

# Preguntas del análisis exploratorio, resueltas con Spark

Las mismas preguntas del notebook de Pandas, ahora con RDDs.
La función `promedio_por_clase` calcula ambos promedios en una sola pasada con `reduceByKey`,
en lugar de filtrar el RDD dos veces.

## Pregunta 1: proporción de URLs de phishing

In [13]:
total, phishing = rdd.map(lambda fila: (1, 1 if fila.label == 0 else 0)).reduce(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)
print(f"Proporción de phishing: {phishing / total:.2%} ({phishing} de {total})")

Proporción de phishing: 42.79% (90801 de 212216)


## Pregunta 2: largo y composición de la URL

In [14]:
mostrar_promedios([
    "URLLength", "DomainLength",
    "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL",
    "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL",
])

URLLength                    phishing: 45.6491   legítimas: 26.2307
DomainLength                 phishing: 24.4601   legítimas: 19.2307
NoOfLettersInURL             phishing: 28.0810   legítimas: 12.9349
LetterRatioInURL             phishing: 0.5682   legítimas: 0.4767
NoOfDegitsInURL              phishing: 4.2990   legítimas: 0.0503
DegitRatioInURL              phishing: 0.0641   legítimas: 0.0021
NoOfOtherSpecialCharsInURL   phishing: 3.7922   legítimas: 1.2453
SpacialCharRatioInURL        phishing: 0.0833   legítimas: 0.0484


### Respuesta

Usando las columnas originales del dataset, las URLs de phishing son más largas (45,7 caracteres contra 26,2)
y tienen más letras, más dígitos y más caracteres especiales, tanto en cantidad como en proporción.
La diferencia más marcada vuelve a ser la de dígitos: 4,3 por URL en phishing contra 0,05 en legítimas.

Los valores difieren de los del notebook de Pandas porque allí se recalcularon los conteos sobre la URL completa.

## Pregunta 3: self, empty y external references

In [15]:
mostrar_promedios(["NoOfSelfRef", "NoOfEmptyRef", "NoOfExternalRef"], formato=".2f")

NoOfSelfRef                  phishing: 0.49   legítimas: 113.43
NoOfEmptyRef                 phishing: 0.15   legítimas: 4.07
NoOfExternalRef              phishing: 1.13   legítimas: 85.23


### Respuesta

Las páginas legítimas tienen muchas más referencias de los tres tipos (113,4 self, 4,1 empty y 85,2 external en promedio)
que las de phishing (0,5, 0,15 y 1,1). Las páginas de phishing tienen estructuras mucho más simples.

## Pregunta 4: Bank, Pay y Crypto

In [16]:
# Son binarias: el promedio es la proporción de páginas que las tienen
mostrar_promedios(["Bank", "Pay", "Crypto"], formato=".2%")

Bank                         phishing: 5.45%   legítimas: 18.15%
Pay                          phishing: 6.07%   legítimas: 36.89%
Crypto                       phishing: 0.62%   legítimas: 3.66%


### Respuesta

| Variable | En phishing | En legítimas |
|---|---|---|
| Bank | 5,45% | 18,15% |
| Pay | 6,07% | 36,89% |
| Crypto | 0,62% | 3,66% |

Las tres variables son bastante más frecuentes en páginas legítimas.

## Pregunta 5: campos de contraseña, campos ocultos y botones de submit

In [17]:
mostrar_promedios(["HasPasswordField", "HasHiddenFields", "HasSubmitButton"], formato=".2%")

HasPasswordField             phishing: 5.43%   legítimas: 13.85%
HasHiddenFields              phishing: 9.35%   legítimas: 59.05%
HasSubmitButton              phishing: 8.53%   legítimas: 66.08%


### Respuesta

| Característica | En phishing | En legítimas |
|---|---|---|
| Campo de contraseña | 5,43% | 13,85% |
| Campos ocultos | 9,35% | 59,05% |
| Botón de submit | 8,53% | 66,08% |

Las tres características son mucho más comunes en páginas legítimas.

In [18]:
spark.stop()